# IBM AML (HI-Medium) - Model A: Financial Features Baseline

**Study case:** Anti-Money Laundering detection on the IBM HI-Medium synthetic
transaction dataset using account-level financial features and XGBoost.

**Approach (Model A):** Tabular features derived from individual account behaviour
-- no graph structure, no community detection. Pure financial signal.

**Feature groups (43 total):**
- Transaction-level (13): amounts, log-amounts, flags (cross-border, same-bank, self-loop, night, weekend)
- Sender account aggregates (17): volume, frequency, structuring patterns, fan-out, concentration
- Receiver account aggregates (8): volume, frequency, fan-in, coefficient of variation
- Categorical (5): From Bank, To Bank, Payment Format, Receiving Currency, Payment Currency

**Next step:** Compare with Graph Neural Networks (GNN) -- see Section 15.

## 0) Setup

In [ ]:
import os, glob, gc, json, shutil, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
print('xgboost version:', xgb.__version__)

## 1) Paths

Set `AML_DATA_ROOT` env var to your `data/raw` directory, or place data under `data/raw/`.

In [ ]:
DATASET = 'HI-Medium'
FORCE_RETRAIN = False  # Set True to retrain even if saved model exists

_env = os.environ.get('AML_DATA_ROOT', '')
if _env and os.path.isdir(_env):
    BASE = _env
else:
    _nb_dir = os.path.abspath(os.getcwd())
    BASE = None
    for _candidate in [
        os.path.join(_nb_dir, 'data', 'raw'),
        os.path.join(_nb_dir, '..', 'data', 'raw'),
        os.path.join(_nb_dir, '..', '..', 'data', 'raw'),
    ]:
        if os.path.isdir(_candidate):
            BASE = os.path.abspath(_candidate)
            break
    if BASE is None:
        raise FileNotFoundError('Cannot locate data/raw/. Set AML_DATA_ROOT env var.')

TRANS_PATH  = os.path.join(BASE, f'{DATASET}_Trans.csv')
PARQUET_DIR = os.path.join(BASE, f'{DATASET}_days_parquet')
OUT_DIR     = os.path.join(BASE, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(TRANS_PATH), f'File not found: {TRANS_PATH}'
print(f'CSV    : {TRANS_PATH}')
print(f'Parquet: {PARQUET_DIR}')
print(f'Outputs: {OUT_DIR}')

## 2) Parquet partitioning

Partition the raw CSV by day. Set `REBUILD_PARQUET = True` to force rebuild.

In [ ]:
REBUILD_PARQUET = False
CHUNKSIZE = 2_000_000

DTYPES = {
    'From Bank': 'int32', 'To Bank': 'int32',
    'Account': 'string', 'Account.1': 'string',
    'Amount Received': 'float32', 'Receiving Currency': 'string',
    'Amount Paid': 'float32', 'Payment Currency': 'string',
    'Payment Format': 'string', 'Is Laundering': 'int8', 'Timestamp': 'string',
}

def add_tx_features(df):
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y/%m/%d %H:%M', errors='coerce')
    df = df.dropna(subset=['Timestamp'])
    df['from_acct_id'] = df['From Bank'].astype('string') + '_' + df['Account'].astype('string')
    df['to_acct_id']   = df['To Bank'].astype('string') + '_' + df['Account.1'].astype('string')
    df['cross_border'] = (df['Receiving Currency'] != df['Payment Currency']).astype('int8')
    df['same_bank']    = (df['From Bank'] == df['To Bank']).astype('int8')
    df['self_loop']    = (df['from_acct_id'] == df['to_acct_id']).astype('int8')
    df['hour']         = df['Timestamp'].dt.hour.astype('int16')
    df['weekday']      = df['Timestamp'].dt.weekday.astype('int16')
    df['is_weekend']   = (df['weekday'] >= 5).astype('int8')
    df['is_night']     = ((df['hour'] >= 20) | (df['hour'] <= 6)).astype('int8')
    df['amt_paid_log1p'] = np.log1p(np.clip(df['Amount Paid'].values, 0, None)).astype('float32')
    df['amt_recv_log1p'] = np.log1p(np.clip(df['Amount Received'].values, 0, None)).astype('float32')
    diff = df['Amount Paid'].astype('float32') - df['Amount Received'].astype('float32')
    df['amt_diff'] = diff
    df['amt_diff_abs_ratio'] = (np.abs(diff) / (np.abs(df['Amount Received']) + 1e-6)).astype('float32')
    df['day'] = df['Timestamp'].dt.floor('D').astype('datetime64[ns]')
    return df

def build_day_parquets(csv_path, out_dir, chunksize=2_000_000):
    if os.path.exists(out_dir): shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    part_idx = 0
    for chunk in pd.read_csv(csv_path, dtype=DTYPES, chunksize=chunksize):
        chunk = add_tx_features(chunk)
        for day, g in chunk.groupby('day', sort=False):
            day_str = pd.Timestamp(day).strftime('%Y-%m-%d')
            day_dir = os.path.join(out_dir, f'day={day_str}')
            os.makedirs(day_dir, exist_ok=True)
            g.drop(columns=['day']).to_parquet(
                os.path.join(day_dir, f'part_{part_idx:06d}.parquet'), index=False)
        part_idx += 1
    print(f'Finished. Parts: {part_idx}')

if REBUILD_PARQUET or not os.path.exists(PARQUET_DIR) or \
   len(glob.glob(os.path.join(PARQUET_DIR, 'day=*'))) == 0:
    print('Building Parquet partitions...')
    build_day_parquets(TRANS_PATH, PARQUET_DIR, chunksize=CHUNKSIZE)
else:
    print('Using existing Parquet partitions.')

## 3) Temporal split (60 / 20 / 20)

Strict chronological split. Days with < 5,000 tx and > 20% fraud rate excluded
(IBM synthetic residual tail, Sep 17-28).

In [ ]:
day_dirs = sorted(glob.glob(os.path.join(PARQUET_DIR, 'day=*')))
days = [os.path.basename(d).split('day=')[1] for d in day_dirs]

def day_stats(day):
    files = glob.glob(os.path.join(PARQUET_DIR, f'day={day}', '*.parquet'))
    col = pd.concat([pd.read_parquet(f, columns=['Is Laundering']) for f in files])
    return {'day': day, 'n': len(col), 'pos_rate': float(col['Is Laundering'].mean())}

stats = pd.DataFrame([day_stats(d) for d in days])
print(stats.to_string(index=False))

mask_normal = ~((stats['n'] < 5_000) & (stats['pos_rate'] > 0.20))
normal_days = stats.loc[mask_normal, 'day'].tolist()
excluded    = stats.loc[~mask_normal, 'day'].tolist()
print(f'\nExcluded ({len(excluded)}): {excluded[0]} to {excluded[-1]}')
print(f'Normal   ({len(normal_days)}): {normal_days[0]} to {normal_days[-1]}')

n = len(normal_days)
n_test = max(2, round(n * 0.20))
n_val  = max(1, round(n * 0.20))
test_days  = normal_days[-n_test:]
val_days   = normal_days[-(n_test + n_val):-n_test]
train_days = normal_days[:-(n_test + n_val)]
print(f'\nTrain: {train_days[0]} -> {train_days[-1]} ({len(train_days)} days)')
print(f'Val  : {val_days[0]} -> {val_days[-1]} ({len(val_days)} days)')
print(f'Test : {test_days[0]} -> {test_days[-1]} ({len(test_days)} days)')

def read_days(days_list):
    files = []
    for d in days_list:
        files.extend(glob.glob(os.path.join(PARQUET_DIR, f'day={d}', '*.parquet')))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

train_df = read_days(train_days)
val_df   = read_days(val_days)
test_df  = read_days(test_days)

TARGET = 'Is Laundering'
print(f'\nSizes  : train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
print(f'Pos rate: train={train_df[TARGET].mean():.5f}  val={val_df[TARGET].mean():.5f}  test={test_df[TARGET].mean():.5f}')

## 4) Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
counts = all_df[TARGET].value_counts()
axes[0].bar(['Legitimate', 'Laundering'], [counts[0], counts[1]], color=['steelblue', 'tomato'])
axes[0].set_title('Class Balance (full dataset)')
axes[0].set_ylabel('Transactions')
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v*1.01, f'{v:,}\n({v/len(all_df):.3%})', ha='center', va='bottom', fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))
axes[0].grid(axis='y', alpha=0.4)

split_map = dict([(d,'train') for d in train_days]+[(d,'val') for d in val_days]+[(d,'test') for d in test_days])
stats2 = stats[stats['day'].isin(normal_days)].copy()
stats2['split'] = stats2['day'].map(split_map)
colors_split = {'train':'steelblue','val':'darkorange','test':'tomato'}
for sp in ['train','val','test']:
    sub = stats2[stats2['split']==sp]
    axes[1].bar(sub['day'], sub['n'], label=sp.capitalize(), color=colors_split[sp], alpha=0.85)
axes[1].set_title('Daily Transaction Volume by Split')
axes[1].set_ylabel('Transactions')
axes[1].set_xlabel('Date')
axes[1].tick_params(axis='x', rotation=45)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))
axes[1].legend(); axes[1].grid(axis='y', alpha=0.4)

for sp in ['train','val','test']:
    sub = stats2[stats2['split']==sp]
    axes[2].plot(sub['day'], sub['pos_rate']*100, 'o-', label=sp.capitalize(), color=colors_split[sp])
axes[2].set_title('Daily Fraud Rate by Split')
axes[2].set_ylabel('Fraud rate (%)')
axes[2].set_xlabel('Date'); axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(); axes[2].grid(alpha=0.4)

plt.suptitle('Dataset Overview -- IBM HI-Medium', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_dataset_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
del all_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fraud_mask = train_df[TARGET]==1
legit_mask = train_df[TARGET]==0

log_fraud = np.log10(train_df.loc[fraud_mask,'Amount Paid'].clip(lower=1))
log_legit = np.log10(train_df.loc[legit_mask,'Amount Paid'].clip(lower=1))
axes[0].hist(log_legit, bins=60, alpha=0.5, density=True, color='steelblue', label='Legitimate')
axes[0].hist(log_fraud, bins=60, alpha=0.6, density=True, color='tomato',    label='Laundering')
axes[0].set_xlabel('log10(Amount Paid)'); axes[0].set_ylabel('Density')
axes[0].set_title('Amount Distribution: Fraud vs Legitimate')
axes[0].legend(); axes[0].grid(alpha=0.4)

pf_stats = train_df.groupby('Payment Format', observed=True)[TARGET].agg(['mean','sum','count'])
pf_stats.columns = ['fraud_rate','n_fraud','n_total']
pf_stats = pf_stats.sort_values('fraud_rate', ascending=True)
avg_rate = train_df[TARGET].mean()
bar_colors = ['tomato' if r > avg_rate*3 else 'steelblue' for r in pf_stats['fraud_rate']]
axes[1].barh(pf_stats.index, pf_stats['fraud_rate']*100, color=bar_colors)
axes[1].axvline(avg_rate*100, color='black', ls='--', lw=1.2, label=f'Avg ({avg_rate*100:.3f}%)')
axes[1].set_xlabel('Fraud rate (%)'); axes[1].set_title('Fraud Rate by Payment Format')
axes[1].legend(); axes[1].grid(axis='x', alpha=0.4)
for i,(pf,row) in enumerate(pf_stats.iterrows()):
    axes[1].text(row['fraud_rate']*100+0.002, i, f"{row['fraud_rate']*100:.3f}%", va='center', fontsize=8)

hour_stats = train_df.groupby('hour')[TARGET].mean()*100
axes[2].plot(hour_stats.index, hour_stats.values, 'o-', color='darkorange', lw=2)
axes[2].axhline(avg_rate*100, color='black', ls='--', lw=1.2, label='Average')
axes[2].fill_between([20,24], 0, hour_stats.max()*1.1, alpha=0.1, color='navy', label='Night window')
axes[2].fill_between([0,6],   0, hour_stats.max()*1.1, alpha=0.1, color='navy')
axes[2].set_xlabel('Hour of day'); axes[2].set_ylabel('Fraud rate (%)')
axes[2].set_title('Fraud Rate by Hour of Day'); axes[2].legend(); axes[2].grid(alpha=0.4)

plt.suptitle('Fraud Signal Distribution -- Training Set', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_fraud_signals.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Payment Format fraud rates (training set):')
print(pf_stats.to_string())

### 4a) Payment Format deep-dive

Payment Format is the single most impactful categorical feature. Understanding *why* requires
examining how each format correlates with other fraud indicators.

| Format | Description | Typical use |
|--------|-------------|-------------|
| **ACH** | Automated Clearing House | Bulk payroll, direct deposit, bill pay -- batch processing with less real-time scrutiny |
| **Wire** | Wire transfer | Large, time-sensitive B2B payments -- heavily regulated, high compliance monitoring |
| **Cheque** | Paper check | Traditional retail payments -- slower clearing cycle |
| **Credit Card** | Card network payment | Consumer retail transactions |
| **Cash** | Cash equivalent | Anonymity-prone, common in smurfing |
| **Bitcoin** | Cryptocurrency | Pseudonymous, cross-border, less regulated |
| **Reinvestment** | Internal reinvestment | Returns within financial products |

**Key finding from training data:** ACH has a fraud rate of **6.28%** vs the dataset average of **0.088%** -- approximately **71x higher**. Wire transfers and Reinvestment show 0% fraud in training, suggesting launderers deliberately avoid heavily-monitored channels.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Volume vs fraud count by PF
pf_full = pf_stats.sort_values('n_total', ascending=True)
x = range(len(pf_full))
bar1 = axes[0].barh([i-0.2 for i in x], pf_full['n_total']/1e6, 0.35, label='Total volume (M)', color='steelblue', alpha=0.7)
ax2 = axes[0].twiny()
ax2.barh([i+0.2 for i in x], pf_full['n_fraud'], 0.35, label='Fraud count', color='tomato', alpha=0.7)
axes[0].set_yticks(list(x)); axes[0].set_yticklabels(pf_full.index, fontsize=9)
axes[0].set_xlabel('Transaction volume (millions)'); ax2.set_xlabel('Fraud count', color='tomato')
axes[0].set_title('Volume vs Fraud Count by Payment Format')
axes[0].legend(loc='lower right', fontsize=8); ax2.legend(loc='center right', fontsize=8)
axes[0].grid(axis='x', alpha=0.3)

# 2) Fraud rate multiplier (relative to average)
pf_mult = pf_stats.copy()
pf_mult['multiplier'] = pf_mult['fraud_rate'] / avg_rate
pf_mult = pf_mult.sort_values('multiplier', ascending=True)
mult_colors = ['tomato' if m>5 else ('darkorange' if m>1 else 'steelblue') for m in pf_mult['multiplier']]
axes[1].barh(pf_mult.index, pf_mult['multiplier'], color=mult_colors)
axes[1].axvline(1.0, color='black', ls='--', lw=1.5, label='1x (average)')
axes[1].set_xlabel('Fraud rate multiplier (vs dataset average)')
axes[1].set_title('Fraud Rate Relative to Dataset Average')
axes[1].legend(); axes[1].grid(axis='x', alpha=0.4)
for i,(pf,row) in enumerate(pf_mult.iterrows()):
    axes[1].text(row['multiplier']+0.2, i, f'{row["multiplier"]:.1f}x', va='center', fontsize=9)

# 3) Cross-border rate by Payment Format
pf_cb = train_df.groupby('Payment Format', observed=True)['cross_border'].mean()*100
pf_cb = pf_cb.sort_values(ascending=True)
axes[2].barh(pf_cb.index, pf_cb.values, color='mediumpurple', alpha=0.8)
axes[2].set_xlabel('Cross-border transaction rate (%)')
axes[2].set_title('Cross-border Rate by Payment Format')
axes[2].grid(axis='x', alpha=0.4)
for i,(pf,v) in enumerate(pf_cb.items()):
    axes[2].text(v+0.3, i, f'{v:.1f}%', va='center', fontsize=9)

plt.suptitle('Payment Format Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_pf_deepdive.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print correlation of ACH with other features
ach_mask = train_df['Payment Format'] == 'ACH'
print('ACH transactions summary:')
print(f'  Count: {ach_mask.sum():,} ({ach_mask.mean():.1%} of training)')
print(f'  Fraud rate: {train_df.loc[ach_mask, TARGET].mean():.4%}')
print(f'  Cross-border rate: {train_df.loc[ach_mask, "cross_border"].mean():.1%}')
print(f'  Same-bank rate: {train_df.loc[ach_mask, "same_bank"].mean():.1%}')
print(f'  Self-loop rate: {train_df.loc[ach_mask, "self_loop"].mean():.1%}')
print(f'  Avg amount: ${train_df.loc[ach_mask, "Amount Paid"].mean():,.0f}')
print(f'\nNon-ACH comparison:')
nach = ~ach_mask
print(f'  Fraud rate: {train_df.loc[nach, TARGET].mean():.4%}')
print(f'  Cross-border rate: {train_df.loc[nach, "cross_border"].mean():.1%}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cb_fraud = train_df.groupby('cross_border')[TARGET].mean()
axes[0].bar(['Same currency','Cross-border'], cb_fraud.values*100, color=['steelblue','tomato'])
axes[0].set_title('Fraud Rate: Cross-border vs Same-currency')
axes[0].set_ylabel('Fraud rate (%)')
for i,v in enumerate(cb_fraud.values*100): axes[0].text(i, v+0.002, f'{v:.3f}%', ha='center', fontsize=10)
axes[0].grid(axis='y', alpha=0.4)

sl_stats = train_df.groupby('self_loop')[TARGET].mean()*100
axes[1].bar(['Normal tx','Self-loop'], sl_stats.values, color=['steelblue','darkorange'])
axes[1].set_title('Fraud Rate: Self-loop Transactions')
axes[1].set_ylabel('Fraud rate (%)')
for i,v in enumerate(sl_stats.values): axes[1].text(i, v+0.002, f'{v:.3f}%', ha='center', fontsize=10)
axes[1].grid(axis='y', alpha=0.4)

day_type = train_df.groupby('is_weekend')[TARGET].mean()*100
axes[2].bar(['Weekday','Weekend'], day_type.values, color=['steelblue','mediumpurple'])
axes[2].set_title('Fraud Rate: Weekday vs Weekend')
axes[2].set_ylabel('Fraud rate (%)')
for i,v in enumerate(day_type.values): axes[2].text(i, v+0.001, f'{v:.3f}%', ha='center', fontsize=10)
axes[2].grid(axis='y', alpha=0.4)

plt.suptitle('Fraud Rate by Transaction Flags -- Training Set', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eda_tx_flags.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5) Feature engineering -- Account-level financial variables

All aggregates are computed **only on the training set** to avoid leakage.
Unseen accounts in val/test receive training-mean imputation.

### Sender features (17)

| Group | Features |
|-------|----------|
| Volume | `total_sent`, `amt_sent_mean`, `amt_sent_std`, `amt_sent_max`, `amt_sent_median` |
| Frequency | `n_tx_sent`, `n_counterparties_sent` |
| Behaviour | `pct_cross_border_sent`, `pct_self_loop`, `pct_night_sent`, `pct_weekend_sent` |
| Typology | `pct_large_tx_sent`, `pct_round_1k_sent`, `pct_structured_sent` |
| Structural | `concentration_dest` (HHI), `fanout_ratio`, `amt_sent_cv` |

### Receiver features (8)

| Group | Features |
|-------|----------|
| Volume | `total_received`, `amt_recv_mean`, `amt_recv_std`, `amt_recv_max` |
| Frequency | `n_tx_received`, `n_counterparties_recv` |
| Structural | `fanin_ratio`, `amt_recv_cv` |

In [ ]:
LARGE_TX_THR = float(np.percentile(train_df['Amount Paid'].dropna().values, 90))

def _structuring_flag(amt):
    flag = pd.Series(False, index=amt.index)
    for thr in [1_000,5_000,10_000,50_000,100_000,500_000,1_000_000]:
        flag = flag | ((amt >= 0.95*thr) & (amt < thr))
    return flag.astype('int8')

def compute_sender_features(df):
    df = df.copy()
    df['is_large']      = (df['Amount Paid'] > LARGE_TX_THR).astype('int8')
    df['is_round_1k']   = ((df['Amount Paid'] % 1_000) < 1.0).astype('int8')
    df['is_structured'] = _structuring_flag(df['Amount Paid'])
    agg = df.groupby('from_acct_id', sort=False).agg(
        total_sent=('Amount Paid','sum'), n_tx_sent=('Amount Paid','count'),
        amt_sent_mean=('Amount Paid','mean'), amt_sent_std=('Amount Paid','std'),
        amt_sent_max=('Amount Paid','max'), amt_sent_median=('Amount Paid','median'),
        n_counterparties_sent=('to_acct_id','nunique'),
        cross_border_sum=('cross_border','sum'), self_loop_sum=('self_loop','sum'),
        night_sum=('is_night','sum'), weekend_sum=('is_weekend','sum'),
        large_tx_sum=('is_large','sum'), round_1k_sum=('is_round_1k','sum'),
        structured_sum=('is_structured','sum'),
        _dest_list=('to_acct_id', lambda x: x.tolist()),
    ).reset_index()
    n = agg['n_tx_sent'].clip(lower=1)
    for col,raw in [('pct_cross_border_sent','cross_border_sum'),('pct_self_loop','self_loop_sum'),
                    ('pct_night_sent','night_sum'),('pct_weekend_sent','weekend_sum'),
                    ('pct_large_tx_sent','large_tx_sum'),('pct_round_1k_sent','round_1k_sum'),
                    ('pct_structured_sent','structured_sum')]:
        agg[col] = agg[raw] / n
    from collections import Counter
    def hhi(dl):
        if len(dl)<=1: return 1.0
        c=Counter(dl); t=sum(c.values())
        return sum((v/t)**2 for v in c.values())
    agg['concentration_dest'] = agg['_dest_list'].apply(hhi).astype('float32')
    agg['fanout_ratio']        = agg['n_counterparties_sent'] / n
    agg['amt_sent_cv']         = agg['amt_sent_std'].fillna(0) / (agg['amt_sent_mean'] + 1e-6)
    drop_cols = ['cross_border_sum','self_loop_sum','night_sum','weekend_sum',
                 'large_tx_sum','round_1k_sum','structured_sum','_dest_list']
    agg = agg.drop(columns=drop_cols)
    agg['amt_sent_std'] = agg['amt_sent_std'].fillna(0)
    return agg

def compute_receiver_features(df):
    agg = df.groupby('to_acct_id', sort=False).agg(
        total_received=('Amount Received','sum'), n_tx_received=('Amount Received','count'),
        amt_recv_mean=('Amount Received','mean'), amt_recv_std=('Amount Received','std'),
        amt_recv_max=('Amount Received','max'), n_counterparties_recv=('from_acct_id','nunique'),
    ).reset_index()
    n = agg['n_tx_received'].clip(lower=1)
    agg['fanin_ratio']  = agg['n_counterparties_recv'] / n
    agg['amt_recv_cv']  = agg['amt_recv_std'].fillna(0) / (agg['amt_recv_mean'] + 1e-6)
    agg['amt_recv_std'] = agg['amt_recv_std'].fillna(0)
    return agg

print('Computing sender features...')
sender_feats   = compute_sender_features(train_df)
print(f'  Unique senders: {len(sender_feats):,}')
print('Computing receiver features...')
receiver_feats = compute_receiver_features(train_df)
print(f'  Unique receivers: {len(receiver_feats):,}')

SENDER_FEAT   = [c for c in sender_feats.columns   if c != 'from_acct_id']
RECEIVER_FEAT = [c for c in receiver_feats.columns if c != 'to_acct_id']
print(f'Sender features ({len(SENDER_FEAT)}); Receiver features ({len(RECEIVER_FEAT)})')

In [ ]:
sender_fill   = sender_feats[SENDER_FEAT].mean().to_dict()
receiver_fill = receiver_feats[RECEIVER_FEAT].mean().to_dict()

def attach_account_features(df):
    out = df.merge(sender_feats,   on='from_acct_id', how='left')
    out = out.merge(receiver_feats, on='to_acct_id',   how='left')
    out[SENDER_FEAT]   = out[SENDER_FEAT].fillna(sender_fill)
    out[RECEIVER_FEAT] = out[RECEIVER_FEAT].fillna(receiver_fill)
    return out

train_df = attach_account_features(train_df)
val_df   = attach_account_features(val_df)
test_df  = attach_account_features(test_df)

train_senders   = set(sender_feats['from_acct_id'])
train_receivers = set(receiver_feats['to_acct_id'])
for sname, df in [('val',val_df),('test',test_df)]:
    df['sender_seen']   = df['from_acct_id'].isin(train_senders).astype('int8')
    df['receiver_seen'] = df['to_acct_id'].isin(train_receivers).astype('int8')
    df['both_seen']     = (df['sender_seen'] & df['receiver_seen']).astype('int8')
    print(f'{sname}: both_seen={df["both_seen"].mean():.1%}  '
          f'sender_unseen={1-df["sender_seen"].mean():.1%}  '
          f'receiver_unseen={1-df["receiver_seen"].mean():.1%}')

### 5a) Feature distributions: fraud vs legitimate

In [ ]:
key_feats = ['pct_structured_sent','concentration_dest','fanout_ratio',
             'pct_cross_border_sent','n_tx_sent','pct_night_sent']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

f_mask = train_df[TARGET]==1
l_mask = train_df[TARGET]==0

for ax, col in zip(axes, key_feats):
    v_fraud = train_df.loc[f_mask,col].dropna()
    v_legit = train_df.loc[l_mask,col].dropna()
    use_log = col in ('n_tx_sent','concentration_dest')
    if use_log:
        v_fraud = np.log10(v_fraud.clip(lower=1e-6))
        v_legit = np.log10(v_legit.clip(lower=1e-6))
        xlabel = f'log10({col})'
    else:
        xlabel = col
    lo = min(v_fraud.quantile(0.01), v_legit.quantile(0.01))
    hi = max(v_fraud.quantile(0.99), v_legit.quantile(0.99))
    bins = np.linspace(lo, hi, 50)
    ax.hist(v_legit, bins=bins, alpha=0.5, density=True, color='steelblue', label='Legitimate')
    ax.hist(v_fraud, bins=bins, alpha=0.6, density=True, color='tomato',    label='Laundering')
    ax.set_xlabel(xlabel, fontsize=8); ax.set_ylabel('Density', fontsize=8)
    ax.set_title(col, fontsize=9); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Feature Distributions: Laundering vs Legitimate (train)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'eda_feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

### 5b) Feature correlation matrix

Pearson correlations of the engineered financial features with each other and with the
fraud label. Only a sample of the training data is used for efficiency.

**Reading the matrix:** Values close to +1/-1 indicate strong co-linearity (potential redundancy).
The right-most column (correlation with `Is Laundering`) shows direct fraud signal strength.

In [ ]:
# Sample to keep heatmap tractable
_n = min(100_000, len(train_df))
_idx = np.random.choice(len(train_df), _n, replace=False)
_sample = train_df.iloc[_idx].copy()

# Numerical features only (exclude string/category)
num_feats = TX_NUM if 'TX_NUM' in dir() else [
    'Amount Received','Amount Paid','amt_recv_log1p','amt_paid_log1p',
    'amt_diff','amt_diff_abs_ratio','cross_border','same_bank','self_loop',
    'hour','weekday','is_weekend','is_night'
]
# We need TX_NUM defined -- will be set in feat matrix cell, so define it here if not already
TX_NUM = ['Amount Received','Amount Paid','amt_recv_log1p','amt_paid_log1p',
          'amt_diff','amt_diff_abs_ratio','cross_border','same_bank','self_loop',
          'hour','weekday','is_weekend','is_night']

corr_cols = SENDER_FEAT + RECEIVER_FEAT + ['cross_border','same_bank','self_loop',
                                            'amt_diff_abs_ratio','is_night','is_weekend']
# Correlation with fraud label -- top features
corr_with_fraud = _sample[corr_cols + [TARGET]].corr()[TARGET].drop(TARGET)
corr_with_fraud = corr_with_fraud.reindex(corr_with_fraud.abs().sort_values(ascending=False).index)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 1) Correlation with fraud label
colors_corr = ['tomato' if v > 0 else 'steelblue' for v in corr_with_fraud.values]
axes[0].barh(range(len(corr_with_fraud)), corr_with_fraud.values[::-1], color=colors_corr[::-1])
axes[0].set_yticks(range(len(corr_with_fraud)))
axes[0].set_yticklabels(corr_with_fraud.index[::-1], fontsize=8)
axes[0].set_xlabel('Pearson correlation with Is Laundering')
axes[0].set_title('Feature Correlation with Fraud Label')
axes[0].axvline(0, color='black', lw=0.8)
axes[0].grid(axis='x', alpha=0.4)

# 2) Top-15 feature correlation heatmap
top15 = corr_with_fraud.abs().sort_values(ascending=False).head(15).index.tolist()
corr_matrix = _sample[top15 + [TARGET]].corr()
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(corr_matrix, mask=mask, ax=axes[1], cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, annot=True, fmt='.2f', annot_kws={'size':7},
            linewidths=0.5, square=True)
axes[1].set_title('Correlation Heatmap -- Top 15 Features + Fraud')
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', rotation=0, labelsize=8)

plt.suptitle('Feature Correlations (training sample, n=100K)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'eda_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Top-10 features most correlated with fraud (|r|):')
for feat, val in corr_with_fraud.head(10).items():
    print(f'  {feat:<35s} r = {val:+.4f}')

### 5c) Feature matrix

In [ ]:
TX_NUM = ['Amount Received','Amount Paid','amt_recv_log1p','amt_paid_log1p',
          'amt_diff','amt_diff_abs_ratio','cross_border','same_bank','self_loop',
          'hour','weekday','is_weekend','is_night']
CAT_LOW   = ['Receiving Currency','Payment Currency','Payment Format']
CAT_BANKS = ['From Bank','To Bank']

FEAT_COLS       = CAT_BANKS + CAT_LOW + TX_NUM + SENDER_FEAT + RECEIVER_FEAT
FEAT_COLS_NO_PF = [c for c in FEAT_COLS if c != 'Payment Format']

for col in CAT_BANKS:
    for df in [train_df, val_df, test_df]: df[col] = df[col].astype('int32')

for c in CAT_LOW:
    train_df[c] = train_df[c].astype('category')
    cats = train_df[c].cat.categories
    val_df[c]  = pd.Categorical(val_df[c],  categories=cats)
    test_df[c] = pd.Categorical(test_df[c], categories=cats)

X_train = train_df[FEAT_COLS]; y_train = train_df[TARGET].astype('int8').to_numpy()
X_val   = val_df[FEAT_COLS];   y_val   = val_df[TARGET].astype('int8').to_numpy()
X_test  = test_df[FEAT_COLS];  y_test  = test_df[TARGET].astype('int8').to_numpy()

print(f'Feature set: {len(FEAT_COLS)} features')
print(f'  Train : {X_train.shape}  pos={y_train.sum():,} ({y_train.mean():.4%})')
print(f'  Val   : {X_val.shape}')
print(f'  Test  : {X_test.shape}')

## 6) Evaluation utilities

In [ ]:
def eval_ranking(y_true, y_score, k_list=(100,500,1000,5000,10000)):
    pr_auc  = float(average_precision_score(y_true, y_score))
    roc_auc = float(roc_auc_score(y_true, y_score))
    order = np.argsort(-y_score); total_pos = int(y_true.sum())
    out = {'PR_AUC':pr_auc,'ROC_AUC':roc_auc,'TotalPos':total_pos}
    for k in k_list:
        kk = min(k, len(order)); tp = int(y_true[order[:kk]].sum())
        out[f'Rec@{kk}'] = round(tp/max(total_pos,1),4)
        out[f'Prec@{kk}'] = round(tp/kk,4)
    return out

def best_fbeta_threshold(y_true, y_score, beta=1.0):
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    fb = (1+beta**2)*prec*rec/(beta**2*prec+rec+1e-12)
    idx = int(np.argmax(fb[:-1])) if len(thr)>0 else 0
    return float(thr[idx]), float(fb[idx]), float(prec[idx]), float(rec[idx])

def workload_table(y_true, y_score,
                   k_list=(50,100,200,500,1000,2000,5000,10000,20000,50000)):
    order = np.argsort(-y_score); total_pos = int(y_true.sum())
    rows = []
    for k in k_list:
        kk = min(k, len(y_true)); tp = int(y_true[order[:kk]].sum())
        rows.append({'K':kk,'TP':tp,'Recall':round(tp/max(total_pos,1),4),
                     'Precision':round(tp/max(kk,1),4),'FP/TP':round((kk-tp)/max(tp,1),1)})
    return pd.DataFrame(rows)

## 7) Hyperparameter selection

Parameters were found via **Optuna** (30 trials, 90-minute budget, PR-AUC on validation as objective).
Optuna uses Bayesian optimisation (TPE sampler) to efficiently explore the search space.

### Search space and final values

| Parameter | Range searched | Final value | Rationale |
|-----------|---------------|-------------|----------|
| `learning_rate` | 0.01 -- 0.3 | **0.089** | Moderate learning rate allows 2000 trees with early stopping |
| `max_depth` | 4 -- 8 | **6** | Moderate tree depth to balance signal capture vs. overfitting on account IDs |
| `min_child_weight` | 10 -- 200 | **60** | Requires at least 60 fraud-weighted samples per leaf; reduces memorisation |
| `subsample` | 0.5 -- 1.0 | **0.623** | Row sampling adds randomness, reduces variance |
| `colsample_bytree` | 0.4 -- 0.9 | **0.600** | Feature sampling per tree -- prevents individual features from dominating |
| `colsample_bynode` | 0.4 -- 0.9 | **0.729** | Additional per-node sampling |
| `reg_lambda` | 1 -- 30 | **13.27** | Strong L2 regularisation -- critical to prevent overfitting on specific accounts |
| `reg_alpha` | 0 -- 5 | **1.046** | Mild L1 regularisation for sparse feature selection |
| `gamma` | 0 -- 10 | **3.847** | Minimum gain to split a node -- acts as a strong pruning filter on the sparse fraud signal |
| `max_bin` | fixed | **256** | Standard histogram bins |

### Class rebalancing: sqrt(neg/pos)

The imbalance ratio is ~1141:1 (neg/pos). Using the full ratio as `scale_pos_weight` causes
the model to over-recall at the cost of very low precision (too many false alarms).
The square-root (sqrt(1141) ~= 34) is a well-established compromise that:
- Gives sufficient weight to the minority class to learn its patterns
- Avoids score saturation that collapses PR-AUC
- Reduces the val-test gap compared to full-ratio weighting

### Early stopping

Training stops when validation `aucpr` does not improve for 80 consecutive rounds.
This prevents overfitting and saves compute. The model found the best iteration at ~997 trees.

## 8) Train XGBoost

If a saved model exists (`FORCE_RETRAIN = False`), it is loaded directly.
Set `FORCE_RETRAIN = True` in Section 1 to re-train.

In [ ]:
pos = int(y_train.sum())
neg = int((y_train==0).sum())
SPW = np.sqrt(neg/max(pos,1))

PARAMS = dict(
    n_estimators=2000, early_stopping_rounds=80,
    objective='binary:logistic', eval_metric='aucpr',
    tree_method='hist', scale_pos_weight=SPW,
    random_state=RANDOM_STATE, n_jobs=-1, enable_categorical=True,
    learning_rate=0.089, max_depth=6, min_child_weight=60,
    subsample=0.623, colsample_bytree=0.600, colsample_bynode=0.729,
    reg_lambda=13.27, reg_alpha=1.046, max_bin=256, gamma=3.847,
)

model_path = os.path.join(OUT_DIR, f'{DATASET}_model_a.json')
summary_path = os.path.join(OUT_DIR, f'{DATASET}_model_a_summary.json')

if not FORCE_RETRAIN and os.path.exists(model_path):
    print(f'Loading saved model from {model_path}')
    model = xgb.XGBClassifier(**{k:v for k,v in PARAMS.items()
                                  if k not in ('n_estimators','early_stopping_rounds')})
    model.load_model(model_path)
    if os.path.exists(summary_path):
        with open(summary_path) as f: _s = json.load(f)
        model.best_iteration = _s.get('best_iteration', 0)
    print(f'Model loaded. Best iteration: {model.best_iteration}')
else:
    print(f'Training XGBoost -- pos={pos:,}  neg={neg:,}  sqrt-SPW={SPW:.2f}')
    model = xgb.XGBClassifier(**PARAMS)
    model.fit(X_train, y_train, eval_set=[(X_val,y_val)], verbose=50)
    print(f'\nBest iteration : {model.best_iteration}')
    print(f'Best val aucpr : {model.best_score:.5f}')

## 9) Evaluation

### 9a) Ranking metrics

In [ ]:
p_val  = model.predict_proba(X_val)[:,1]
p_test = model.predict_proba(X_test)[:,1]

val_m  = eval_ranking(y_val,  p_val)
test_m = eval_ranking(y_test, p_test)

print('='*70)
print('VALIDATION:')
for k,v in val_m.items():  print(f'  {k:<14}: {v}')
print('\nTEST:')
for k,v in test_m.items(): print(f'  {k:<14}: {v}')
print('='*70)
print(f'\nVal-Test gap: PR-AUC={val_m["PR_AUC"]-test_m["PR_AUC"]:.4f}  '
      f'ROC-AUC={val_m["ROC_AUC"]-test_m["ROC_AUC"]:.4f}')

thr_f1,f1_v,p_f1,r_f1 = best_fbeta_threshold(y_val, p_val, beta=1.0)
thr_f2,f2_v,p_f2,r_f2 = best_fbeta_threshold(y_val, p_val, beta=2.0)
print(f'\nF1-threshold (val): {thr_f1:.4f} -> F1={f1_v:.4f}  P={p_f1:.4f}  R={r_f1:.4f}')
print(f'F2-threshold (val): {thr_f2:.4f} -> F2={f2_v:.4f}  P={p_f2:.4f}  R={r_f2:.4f}')

for name, thr in [('F1',thr_f1),('F2',thr_f2)]:
    y_hat = (p_test >= thr).astype('int8')
    print(f'\nTEST -- Classification Report (threshold {name} = {thr:.4f}):')
    print(classification_report(y_test, y_hat, target_names=['Legitimate','Laundering'], digits=4))

print('Analyst Workload Table (TEST):')
print(workload_table(y_test, p_test).to_string(index=False))

### 9b) Isotonic calibration

> **Note on calibration:** After isotonic calibration, PR-AUC may slightly decrease
> (0.1486 -> 0.1436 in this run). This is expected: isotonic regression reshuffles
> scores to make probabilities accurate, which can marginally hurt ranking.
> The calibrated scores are still more interpretable operationally
> (a score of 0.5 genuinely means ~50% fraud probability).

In [ ]:
iso_reg = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')
iso_reg.fit(p_val, y_val)
p_test_cal = iso_reg.predict(p_test)
p_val_cal  = iso_reg.predict(p_val)

print(f'PR-AUC  raw={average_precision_score(y_test, p_test):.6f}  '
      f'calibrated={average_precision_score(y_test, p_test_cal):.6f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, scores, title in [
    (axes[0], p_test,     'Before calibration (raw)'),
    (axes[1], p_test_cal, 'After calibration (isotonic)'),
]:
    pt, pp = calibration_curve(y_test, scores, n_bins=20, strategy='quantile')
    ax.plot(pp, pt, 'o-', label='Model')
    ax.plot([0,1],[0,1],'r--',label='Perfect')
    ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('True fraction of positives')
    ax.set_title(title); ax.legend(); ax.grid(True)
plt.suptitle('Calibration -- Model A', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_calibration.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nCalibrated score interpretation (TEST):')
for lo,hi in [(0,0.01),(0.01,0.05),(0.05,0.10),(0.10,0.25),(0.25,0.50),(0.50,1.0)]:
    mask = (p_test_cal>=lo)&(p_test_cal<hi)
    if mask.sum()==0: continue
    print(f'  [{lo:.2f},{hi:.2f}): {mask.sum():>10,} tx | {int(y_test[mask].sum()):>5,} fraud | '
          f'true rate: {y_test[mask].mean():.4%}')

### 9c) Evaluation panel (6 plots)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

y_hat_f2 = (p_test >= thr_f2).astype('int8')
ConfusionMatrixDisplay(confusion_matrix(y_test,y_hat_f2),
    display_labels=['Legitimate','Laundering']).plot(ax=axes[0,0], cmap='Blues', colorbar=False)
axes[0,0].set_title(f'Confusion Matrix (F2 threshold={thr_f2:.3f})')

prec_c,rec_c,_ = precision_recall_curve(y_test, p_test)
axes[0,1].plot(rec_c, prec_c, lw=2, label=f'XGBoost (PR-AUC={test_m["PR_AUC"]:.4f})')
axes[0,1].axhline(y=float(y_test.mean()), color='r', ls='--', alpha=0.6, label='Random')
axes[0,1].scatter([r_f1],[p_f1],zorder=5,color='green', s=80,marker='^',label='F1 thr')
axes[0,1].scatter([r_f2],[p_f2],zorder=5,color='orange',s=80,marker='v',label='F2 thr')
axes[0,1].set_xlabel('Recall'); axes[0,1].set_ylabel('Precision')
axes[0,1].set_title('PR Curve (TEST)'); axes[0,1].legend(fontsize=8); axes[0,1].grid(True)

fpr,tpr,_ = roc_curve(y_test, p_test)
axes[0,2].plot(fpr,tpr,lw=2,label=f'XGBoost (ROC-AUC={test_m["ROC_AUC"]:.4f})')
axes[0,2].plot([0,1],[0,1],'r--',alpha=0.6,label='Random')
axes[0,2].set_xlabel('FPR'); axes[0,2].set_ylabel('TPR')
axes[0,2].set_title('ROC Curve (TEST)'); axes[0,2].legend(); axes[0,2].grid(True)

axes[1,0].hist(p_test_cal[y_test==0],bins=80,alpha=0.6,density=True,color='steelblue',label='Legitimate')
axes[1,0].hist(p_test_cal[y_test==1],bins=80,alpha=0.6,density=True,color='tomato',   label='Laundering')
axes[1,0].set_xlabel('Calibrated P(laundering)'); axes[1,0].set_ylabel('Density')
axes[1,0].set_title('Score Distribution (calibrated)'); axes[1,0].legend(); axes[1,0].grid(True)

pt,pp = calibration_curve(y_test, p_test_cal, n_bins=20, strategy='quantile')
axes[1,1].plot(pp,pt,'o-',label='Calibrated'); axes[1,1].plot([0,1],[0,1],'r--',label='Perfect')
axes[1,1].set_xlabel('Predicted probability'); axes[1,1].set_ylabel('True fraction')
axes[1,1].set_title('Calibration (post-isotonic)'); axes[1,1].legend(); axes[1,1].grid(True)

feat_names = list(X_train.columns)
imp = pd.Series(model.feature_importances_, index=feat_names).sort_values(ascending=False)
imp.head(20).plot.barh(ax=axes[1,2]); axes[1,2].invert_yaxis()
axes[1,2].set_title('Feature Importance -- gain (top 20)')
axes[1,2].set_xlabel('Importance score')

plt.suptitle('Model A -- Financial Features Baseline', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_eval_6panel.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nTop-20 features:')
for i,(f,v) in enumerate(imp.head(20).items(),1):
    print(f'  {i:2d}. {f:<35s} {v:.4f}')

### 9d) Recall@K -- analyst workload

> **Reading the workload curve:** Each point shows how many fraud cases an analyst
> finds if they review the top-K highest-scored transactions. High precision at low K
> (e.g., 92% at K=50) means the model is very confident in its top alerts -- good for
> prioritisation. Recall grows slowly because fraud is very rare and the model cannot
> identify all cases with high confidence.

In [ ]:
wt = workload_table(y_test, p_test)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(wt['K'], wt['Recall'], 'o-', color='steelblue', lw=2)
axes[0].set_xlabel('Alerts reviewed (K)'); axes[0].set_ylabel('Recall')
axes[0].set_title('Recall@K -- Analyst Coverage'); axes[0].grid(alpha=0.4)
for _,row in wt.iterrows():
    if row['K'] in (100,1000,5000,10000):
        axes[0].annotate(f"K={int(row['K'])}\n{row['Recall']:.1%}",
                         (row['K'],row['Recall']), textcoords='offset points',
                         xytext=(8,4), fontsize=8, color='steelblue')

axes[1].plot(wt['K'], wt['Precision'], 'o-', color='tomato', lw=2)
axes[1].set_xlabel('Alerts reviewed (K)'); axes[1].set_ylabel('Precision (hit rate)')
axes[1].set_title('Precision@K -- Alert Quality'); axes[1].grid(alpha=0.4)
for _,row in wt.iterrows():
    if row['K'] in (100,1000,5000,10000):
        axes[1].annotate(f"K={int(row['K'])}\n{row['Precision']:.1%}",
                         (row['K'],row['Precision']), textcoords='offset points',
                         xytext=(8,4), fontsize=8, color='tomato')

plt.suptitle('Analyst Workload -- Model A (TEST)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_workload_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

### 9e) Memorization diagnostic

> **Key finding:** PR-AUC drops from **0.160** (both accounts seen in training) to **0.075**
> (both unseen). This 2x degradation reveals that the model partially relies on account
> identity memorisation. The large train-val gap in the learning curve confirms this.
> This is the primary motivation for the GNN approach (Section 15).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
segments = {
    'All test':        np.ones(len(y_test), dtype=bool),
    'Both seen':       (test_df['both_seen']==1).values,
    'Sender unseen':   (test_df['sender_seen']==0).values,
    'Receiver unseen': (test_df['receiver_seen']==0).values,
    'Both unseen':     (test_df['both_seen']==0).values,
}
seg_m = {}
for name, mask in segments.items():
    if mask.sum()<10 or y_test[mask].sum()<1:
        seg_m[name] = {'PR_AUC':float('nan'),'n':int(mask.sum()),'pos':0}
    else:
        seg_m[name] = {'PR_AUC':float(average_precision_score(y_test[mask],p_test[mask])),
                       'n':int(mask.sum()),'pos':int(y_test[mask].sum())}
print('PR-AUC by account visibility:')
print(pd.DataFrame(seg_m).T.to_string())

names=list(seg_m.keys()); vals=[seg_m[n]['PR_AUC'] for n in names]
colors=['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']
axes[0].barh(names, vals, color=colors)
axes[0].set_xlabel('PR-AUC'); axes[0].set_title('PR-AUC by Visibility Segment')
axes[0].grid(True, axis='x')
for i,v in enumerate(vals):
    if not np.isnan(v): axes[0].text(v+0.001,i,f'{v:.4f}',va='center',fontsize=9)

pos_mask=y_test==1
seen_p  =pos_mask&(test_df['both_seen']==1).values
unseen_p=pos_mask&(test_df['both_seen']==0).values
if seen_p.sum()>0:   axes[1].hist(p_test[seen_p],  bins=40,alpha=0.6,density=True,label=f'Seen ({seen_p.sum()})',  color='steelblue')
if unseen_p.sum()>0: axes[1].hist(p_test[unseen_p],bins=40,alpha=0.6,density=True,label=f'Unseen ({unseen_p.sum()})',color='tomato')
axes[1].set_xlabel('Score'); axes[1].set_title('Fraud scores: Seen vs Unseen')
axes[1].legend(); axes[1].grid(True)

k_range=[50,100,200,500,1000,2000,5000,10000]
for sname, mask in [('All',np.ones(len(y_test),dtype=bool)),
                    ('Both seen',(test_df['both_seen']==1).values),
                    ('Any unseen',(test_df['both_seen']==0).values)]:
    if mask.sum()<10 or y_test[mask].sum()<1: continue
    tp=y_test[mask].sum(); order=np.argsort(-p_test[mask])
    recalls=[int(y_test[mask][order[:min(k,mask.sum())]].sum())/max(tp,1) for k in k_range]
    axes[2].plot(k_range,recalls,'o-',label=sname)
axes[2].set_xlabel('K'); axes[2].set_ylabel('Recall')
axes[2].set_title('Recall@K by Visibility'); axes[2].legend(); axes[2].grid(True)

plt.suptitle('Memorization Diagnostic -- Model A', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_memorization.png'), dpi=150, bbox_inches='tight')
plt.show()

### 9f) Learning curve

> **Interpretation:** The train-val gap (~0.28-0.40 PR-AUC) is large and *decreases only slowly*
> as training size grows (0.40 at 5% -> 0.28 at 100%). This indicates **memorisation of account
> identities** rather than true pattern generalisation. The model learns 'this account has been
> involved in fraud' rather than 'accounts with these patterns tend to launder'. Adding more data
> helps modestly, but the structural problem (tabular = no graph awareness) remains.

In [ ]:
fractions=[0.05,0.15,0.40,1.0]
lc_train,lc_val=[],[]
TRAIN_EVAL_CAP=500_000
lc_params={k:v for k,v in PARAMS.items() if k not in ('n_estimators','early_stopping_rounds')}
lc_params['n_estimators']=300
lc_params['early_stopping_rounds']=20

for frac in fractions:
    n_sub=int(len(X_train)*frac)
    idx=np.random.choice(len(X_train),n_sub,replace=False)
    _m=xgb.XGBClassifier(**lc_params)
    _m.fit(X_train.iloc[idx],y_train[idx],eval_set=[(X_val,y_val)],verbose=False)
    eval_n=min(TRAIN_EVAL_CAP,len(idx))
    ptr=_m.predict_proba(X_train.iloc[idx[:eval_n]])[:,1]
    pvl=_m.predict_proba(X_val)[:,1]
    lc_train.append(average_precision_score(y_train[idx[:eval_n]],ptr))
    lc_val.append(average_precision_score(y_val,pvl))
    print(f'  {frac:.0%} ({n_sub:,}): train={lc_train[-1]:.4f}  val={lc_val[-1]:.4f}  gap={lc_train[-1]-lc_val[-1]:.4f}')
    del _m; gc.collect()

fig,ax=plt.subplots(figsize=(8,5))
sizes=[int(len(X_train)*f) for f in fractions]
ax.plot(sizes,lc_train,'o-',label='Train PR-AUC',     color='steelblue')
ax.plot(sizes,lc_val,  'o-',label='Validation PR-AUC',color='tomato')
ax.set_xlabel('Training set size'); ax.set_ylabel('PR-AUC')
ax.set_title('Learning Curve -- Model A'); ax.legend(); ax.grid(True)
for j in range(len(fractions)):
    ax.annotate(f'gap={lc_train[j]-lc_val[j]:.3f}',
                (sizes[j],(lc_train[j]+lc_val[j])/2),fontsize=7,ha='center',color='gray')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_learning_curve.png'),dpi=150)
plt.show()

## 10) Payment Format ablation

In [ ]:
nopf_model_path = os.path.join(OUT_DIR, f'{DATASET}_model_a_no_pf.json')

if not FORCE_RETRAIN and os.path.exists(nopf_model_path):
    print(f'Loading no-PF model from {nopf_model_path}')
    model_nopf = xgb.XGBClassifier(**{k:v for k,v in PARAMS.items()
                                       if k not in ('n_estimators','early_stopping_rounds')})
    model_nopf.load_model(nopf_model_path)
else:
    print('Training model WITHOUT Payment Format...')
    model_nopf = xgb.XGBClassifier(**PARAMS)
    model_nopf.fit(X_train[FEAT_COLS_NO_PF],y_train,
                   eval_set=[(X_val[FEAT_COLS_NO_PF],y_val)],verbose=100)

p_test_nopf = model_nopf.predict_proba(X_test[FEAT_COLS_NO_PF])[:,1]
m_nopf = eval_ranking(y_test, p_test_nopf)

print('\n'+'='*70)
print('ABLATION: Payment Format')
print('='*70)
print(pd.DataFrame({'With PF':test_m,'Without PF':m_nopf}).T.to_string())
print()
for key in ['PR_AUC','ROC_AUC','Rec@1000','Rec@5000','Rec@10000']:
    pct=(m_nopf[key]-test_m[key])/max(test_m[key],1e-9)*100
    print(f'  {key:<14}: {test_m[key]:.4f} -> {m_nopf[key]:.4f}  ({pct:+.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
p1,r1,_=precision_recall_curve(y_test,p_test)
p2,r2,_=precision_recall_curve(y_test,p_test_nopf)
axes[0].plot(r1,p1,lw=2,        label=f'With PF    (PR-AUC={test_m["PR_AUC"]:.4f})')
axes[0].plot(r2,p2,lw=2,ls='--',label=f'Without PF (PR-AUC={m_nopf["PR_AUC"]:.4f})')
axes[0].axhline(y=float(y_test.mean()),color='r',ls=':',alpha=0.5,label='Random')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('PR Curve: With vs Without Payment Format')
axes[0].legend(); axes[0].grid(True)

imp_nopf=pd.Series(model_nopf.feature_importances_,index=FEAT_COLS_NO_PF).sort_values(ascending=False)
imp_nopf.head(15).plot.barh(ax=axes[1]); axes[1].invert_yaxis()
axes[1].set_title('Feature Importance without Payment Format (top 15)')
axes[1].set_xlabel('Importance score')

plt.suptitle('Payment Format Ablation', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_pf_ablation.png'),dpi=150,bbox_inches='tight')
plt.show()

## 11) SHAP explanations

SHAP decomposes each prediction into per-feature contributions.

In [ ]:
import shap
print(f'SHAP version: {shap.__version__}')

N_SHAP=1_000; rng=np.random.default_rng(RANDOM_STATE)
pos_i=np.where(y_test==1)[0]; neg_i=np.where(y_test==0)[0]
n_p=min(len(pos_i),N_SHAP//2)
shap_idx=np.concatenate([rng.choice(pos_i,n_p,replace=False),rng.choice(neg_i,N_SHAP-n_p,replace=False)])
X_shap=X_test.iloc[shap_idx].reset_index(drop=True)
y_shap=y_test[shap_idx]

X_shap_num=X_shap.copy()
for c in X_shap_num.select_dtypes('category').columns:
    X_shap_num[c]=X_shap_num[c].cat.codes.astype('int32')
fn=list(X_shap_num.columns)
Xarr=X_shap_num.values.astype('float32')
dmat=xgb.DMatrix(Xarr,feature_names=fn)

shap_mat=model.get_booster().predict(dmat,pred_contribs=True)
sv=shap_mat[:,:-1]; bias=float(shap_mat[0,-1])
print(f'SHAP matrix: {sv.shape}  base value: {bias:.4f}')

mean_abs=np.abs(sv).mean(0); t20=np.argsort(-mean_abs)[:20]
fig,ax=plt.subplots(figsize=(8,6))
ax.barh(range(20),mean_abs[t20][::-1])
ax.set_yticks(range(20)); ax.set_yticklabels([fn[i] for i in t20][::-1],fontsize=8)
ax.set_xlabel('Mean |SHAP value|'); ax.set_title('SHAP Global Importance (top 20)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_shap_importance.png'),dpi=150); plt.show()

expl=shap.Explanation(values=sv,base_values=np.full(len(sv),bias),data=Xarr,feature_names=fn)
plt.figure(figsize=(8,8))
shap.plots.beeswarm(expl,max_display=20,show=False)
plt.title('SHAP Beeswarm -- Model A')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_shap_beeswarm.png'),dpi=150); plt.show()

p_sh=model.predict_proba(X_shap)[:,1]
ti=int(np.argmax(p_sh))
print(f'Most suspicious: score={p_sh[ti]:.4f}  true label={y_shap[ti]}')
plt.figure(figsize=(10,6))
shap.plots.waterfall(expl[ti],max_display=15,show=False)
plt.title('SHAP Waterfall -- Most Suspicious Transaction')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_shap_waterfall.png'),dpi=150); plt.show()

pm=y_shap==1; nm=y_shap==0
msp=sv[pm].mean(0); msn=sv[nm].mean(0)
t20i=np.argsort(-np.abs(msp))[:20]
fig,ax=plt.subplots(figsize=(10,7))
x=np.arange(20); w=0.35
ax.barh(x-w/2,msp[t20i],w,label='Laundering',color='tomato',    alpha=0.8)
ax.barh(x+w/2,msn[t20i],w,label='Legitimate',color='steelblue', alpha=0.8)
ax.set_yticks(x); ax.set_yticklabels([fn[i] for i in t20i],fontsize=8)
ax.invert_yaxis(); ax.set_xlabel('Mean SHAP value')
ax.set_title('Mean SHAP: Laundering vs Legitimate')
ax.legend(); ax.grid(True,axis='x')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_shap_comparison.png'),dpi=150); plt.show()

## 12) Risk tiers

Alert tiers are **precision-anchored**: thresholds are calibrated on the validation set
so each tier guarantees a minimum fraction of genuine fraud in the alert queue.

This is operationally meaningful: `Critical` means 'at least 15 out of every 100 alerts
reviewed in this tier are real laundering cases' -- a number a compliance team can plan around.

| Tier | Min. precision | Interpretation |
|------|---------------|----------------|
| **Critical** | >= 15% | ~1 in 7 alerts is genuine fraud -- immediate investigation |
| **High** | >= 5% | ~1 in 20 -- priority review queue |
| **Medium-High** | >= 2% | ~1 in 50 -- standard investigation queue |
| **Medium** | >= 0.5% | ~1 in 200 (~6x baseline) -- secondary screening |
| **Low** | >= 0.2% | ~1 in 500 (~2x baseline) -- automated flag |
| **Very Low** | >= 0.1% | ~1 in 1000 (just above random) -- watchlist only |

In [ ]:
# Precision-anchored tiers: calibrate thresholds on validation set
prec_v, rec_v, thr_v = precision_recall_curve(y_val, p_val_cal)
# prec_v[i] is precision at threshold thr_v[i]; sklearn orders by descending threshold
# thr_v is ascending, prec_v[:-1] lines up with thr_v

tier_defs = [
    ('Critical',    0.15, '#d32f2f', '>=15% genuine fraud -- immediate investigation'),
    ('High',        0.05, '#f57c00', '>=5%  genuine fraud -- priority review'),
    ('Medium-High', 0.02, '#fbc02d', '>=2%  genuine fraud -- standard queue'),
    ('Medium',      0.005,'#388e3c', '>=0.5% genuine fraud (~6x baseline)'),
    ('Low',         0.002,'#0288d1', '>=0.2% genuine fraud (~2x baseline)'),
    ('Very Low',    0.001,'#7b1fa2', '>=0.1% genuine fraud (above random)'),
]

# Find minimum threshold on val that achieves >= target precision
def precision_threshold(prec_arr, thr_arr, target):
    valid = thr_arr[prec_arr[:-1] >= target]
    return float(valid.min()) if len(valid) > 0 else None

tier_thresholds = {name: precision_threshold(prec_v, thr_v, tgt)
                   for name, tgt, *_ in tier_defs}

# Build per-tier metrics on test set
tier_rows = []
total_pos = int(y_test.sum())
prev_thr  = 2.0  # above any calibrated score
for name, tgt, color, desc in tier_defs:
    thr = tier_thresholds[name]
    if thr is None:
        tier_rows.append({'Tier':name,'Min Prec':f'{tgt:.1%}','Threshold':'N/A',
                          'Alerts':0,'Fraud':0,'Precision':0,'Recall(tier)':0,
                          '_color':color,'_desc':desc})
        continue
    mask = (p_test_cal >= thr) & (p_test_cal < prev_thr)
    n_tx = int(mask.sum()); n_fraud = int(y_test[mask].sum())
    tier_rows.append({'Tier':name,'Min Prec':f'{tgt:.1%}','Threshold':round(thr,4),
                      'Alerts':n_tx,'Fraud':n_fraud,
                      'Precision':round(n_fraud/max(n_tx,1),4),
                      'Recall(tier)':round(n_fraud/max(total_pos,1),4),
                      '_color':color,'_desc':desc})
    prev_thr = thr
tier_df = pd.DataFrame(tier_rows)

# Print per-tier summary
print('Risk Tiers -- precision-anchored (val thresholds applied to test):')
print(f"{'Tier':<13} {'Min Prec':>9} {'Thr':>7} {'Alerts':>10} {'Fraud':>7} "
      f"{'Precision':>10} {'Recall':>8}")
print('-'*65)
for _,row in tier_df.iterrows():
    print(f"  {row['Tier']:<12} {row['Min Prec']:>8} {str(row['Threshold']):>7} "
          f"{row['Alerts']:>10,} {row['Fraud']:>7,} "
          f"{row['Precision']:>10.3f} {row['Recall(tier)']:>8.3f}")

# Cumulative recall table (each tier adds on top of the previous)
print(f"\n{'Cumulative (>= tier threshold)':}"
      f"\n{'Tier':<13} {'Threshold':>10} {'Cum. Alerts':>12} {'Cum. Fraud':>11} "
      f"{'Recall':>8} {'Precision':>10}")
print('-'*68)
for name, tgt, *_ in tier_defs:
    thr = tier_thresholds[name]
    if thr is None: continue
    mask = p_test_cal >= thr
    n = int(mask.sum()); f = int(y_test[mask].sum())
    print(f"  {name:<12} {thr:>10.4f} {n:>12,} {f:>11,} "
          f"{f/max(total_pos,1):>8.3f} {f/max(n,1):>10.3f}")

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
valid_tiers = tier_df[tier_df['Threshold'] != 'N/A'].copy()
tier_colors = valid_tiers['_color'].tolist()

# Bar 1: actual precision in tier vs target precision
x = np.arange(len(valid_tiers))
axes[0].bar(x - 0.2, valid_tiers['Precision']*100, 0.35,
            color=tier_colors, alpha=0.85, label='Actual precision')
tgts = [t for _,t,*__ in tier_defs if tier_thresholds[_] is not None]
axes[0].bar(x + 0.2, [t*100 for t in tgts], 0.35,
            color='lightgray', edgecolor='gray', label='Target (min)')
axes[0].set_xticks(x); axes[0].set_xticklabels(valid_tiers['Tier'], rotation=25, fontsize=9)
axes[0].set_ylabel('Precision (%)'); axes[0].set_title('Actual vs Target Precision per Tier')
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.4)
for i,row in valid_tiers.reset_index(drop=True).iterrows():
    axes[0].text(i-0.2, row['Precision']*100+0.1, f"{row['Precision']*100:.1f}%",
                 ha='center', fontsize=7)

# Bar 2: fraud cases per tier
axes[1].bar(valid_tiers['Tier'], valid_tiers['Fraud'], color=tier_colors)
axes[1].set_ylabel('Fraud cases in tier'); axes[1].set_title('Fraud Cases by Risk Tier')
axes[1].tick_params(axis='x', rotation=25); axes[1].grid(axis='y', alpha=0.4)
for i,row in valid_tiers.reset_index(drop=True).iterrows():
    axes[1].text(i, row['Fraud']+3, str(int(row['Fraud'])), ha='center', fontsize=8)

# Plot 3: cumulative recall vs cumulative alert volume
cum_alerts=[]; cum_recall=[]; cum_colors=[]
for name, tgt, color, *_ in tier_defs:
    thr = tier_thresholds[name]
    if thr is None: continue
    mask = p_test_cal >= thr
    cum_alerts.append(int(mask.sum()))
    cum_recall.append(float(y_test[mask].sum()/max(total_pos,1)))
    cum_colors.append(color)
axes[2].plot(cum_alerts, cum_recall, 'o-', color='steelblue', lw=2, zorder=2)
for i,(a,r) in enumerate(zip(cum_alerts,cum_recall)):
    tier_name = [n for n,_,c,*__ in tier_defs if tier_thresholds[n] is not None][i]
    axes[2].annotate(tier_name,(a,r),textcoords='offset points',xytext=(6,3),
                     fontsize=8,color=cum_colors[i])
    axes[2].scatter([a],[r],color=cum_colors[i],s=60,zorder=3)
axes[2].set_xlabel('Cumulative alerts (precision-anchored)')
axes[2].set_ylabel('Cumulative recall')
axes[2].set_title('Recall vs Alert Volume by Tier')
axes[2].grid(alpha=0.4)

plt.suptitle('Risk Tier Analysis -- Model A (Precision-Anchored)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'model_a_risk_tiers.png'),dpi=150,bbox_inches='tight')
plt.show()

## 13) Save artifacts

In [ ]:
scored=test_df[[TARGET]].copy()
scored['score_raw']        = p_test
scored['score_calibrated'] = p_test_cal
scored.sort_values('score_raw',ascending=False).to_csv(
    os.path.join(OUT_DIR,f'{DATASET}_model_a_scores.csv'), index=False)

model.save_model(os.path.join(OUT_DIR,f'{DATASET}_model_a.json'))
model_nopf.save_model(os.path.join(OUT_DIR,f'{DATASET}_model_a_no_pf.json'))

with open(os.path.join(OUT_DIR,f'{DATASET}_model_a_calibrator.pkl'),'wb') as fh:
    pickle.dump(iso_reg, fh)

summary={
    'model':'Model A -- Financial Features Baseline','dataset':DATASET,
    'n_features':len(FEAT_COLS),
    'feature_groups':{'transaction':len(TX_NUM),'categorical':len(CAT_BANKS)+len(CAT_LOW),
                      'sender_agg':len(SENDER_FEAT),'receiver_agg':len(RECEIVER_FEAT)},
    'test_metrics':{k:(float(v) if isinstance(v,(float,__import__('numpy').floating)) else v) for k,v in test_m.items()},
    'test_metrics_no_pf':{k:(float(v) if isinstance(v,(float,__import__('numpy').floating)) else v) for k,v in m_nopf.items()},
    'best_iteration':int(model.best_iteration),'scale_pos_weight':float(SPW),
}
with open(os.path.join(OUT_DIR,f'{DATASET}_model_a_summary.json'),'w') as fh:
    json.dump(summary, fh, indent=2)

print('All artifacts saved to:', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)): print(f'  {f}')

## 14) Why GNNs? -- Motivation and expected improvements

### The limitation of tabular features

Model A treats every transaction as an independent event. The account-level aggregates
(fan-out, concentration, structuring rates) capture *individual account behaviour* over
the training window -- but they are blind to **relationships between accounts**.

### The transaction graph

Every transaction defines a directed edge: **sender account --> receiver account**.
Over the 10-day training period, this creates a graph with:
- ~2 million unique sender accounts (nodes)
- ~1.7 million unique receiver accounts (nodes)
- ~20 million edges (transactions)

Laundering operations are rarely isolated. They form **multi-hop patterns**:

```
Placement:   Dirty cash --> Bank A (ACH, structuring)
Layering:    Bank A --> Bank B --> Bank C (cross-border wire transfers)
Integration: Bank C --> Shell company (legitimate-looking invoices)
```

A tabular model sees each account in isolation. A GNN propagates information
along the graph edges -- a node 3 hops away from a known fraudster gets a higher
score even if its own transaction history looks clean.

### What GNNs add

| Problem | Tabular Model A | GNN |
|---------|-----------------|-----|
| New account (unseen in training) | Mean imputation, PR-AUC drops 2x | Can score via connected known fraudsters |
| Multi-hop layering | Only 1-hop aggregates (fan-out) | k-hop message passing |
| Community detection | None | Learns community membership implicitly |
| Temporal sequence | Aggregates only | Sequential GNN variants |

### Evidence from this study

The memorization diagnostic shows PR-AUC = **0.160** for seen accounts vs **0.075** for unseen
(2x drop). GNNs are expected to close most of this gap by propagating fraud signal through
the graph to unseen nodes.

The IBM benchmark paper (Altman et al., 2023) confirms this: GIN+EU achieves F1 = 54.9
vs MLP = 28.2 on HI-Medium, a **95% improvement** over the tabular baseline.

### Reference

- Altman et al. (2023). *Realistic Synthetic Financial Transactions for Anti-Money Laundering Models.*
  NeurIPS 2023. [arXiv:2306.16424](https://arxiv.org/abs/2306.16424)
- IBM Multi-GNN implementation: [github.com/IBM/Multi-GNN](https://github.com/IBM/Multi-GNN)

## 15) Summary and key findings

### Results (test set, 3 days: Sep 14-16, 2022)

| Metric | Value | Notes |
|--------|-------|-------|
| PR-AUC | **0.1486** | Primary metric for imbalanced detection |
| ROC-AUC | **0.9705** | Misleading on imbalanced data; reported for completeness |
| Val PR-AUC | 0.2655 | Val-Test gap = 0.117 |
| Precision @ 50 alerts | **92%** | High-confidence top alerts |
| Precision @ 100 alerts | **78%** | |
| Recall @ 1,000 alerts | **8.0%** | |
| Recall @ 5,000 alerts | **18.4%** | |
| Best iteration | 997 trees | Early stopping on val PR-AUC |
| PF ablation | **-43.7% PR-AUC** | Removing Payment Format |

### Feature importance findings

Contrary to initial expectation, **`self_loop`** (0.183) and **`same_bank`** (0.177) outrank
**`Payment Format`** (0.119) in gain-based importance.
This suggests that circular transactions (sender = receiver) and intra-bank transfers
are the strongest individual signals -- consistent with shell company and
round-tripping laundering patterns.

Payment Format remains critical for recall at scale (ablation: -43.7% PR-AUC),
primarily driven by **ACH** (Automated Clearing House) which has a fraud rate of
**6.28%** -- approximately **71x** the dataset average.

### Calibration note

PR-AUC slightly decreases after isotonic calibration (0.1486 -> 0.1436).
This is expected: isotonic regression optimises probability accuracy, not ranking.
The calibrated scores are more operationally meaningful
(score 0.5 = ~50% true fraud probability) and are used for the risk tiers.

### Learning curve and memorisation

The large train-val gap (0.28-0.40 PR-AUC) reveals that the model partially
memorises account identity rather than learning purely typology-based patterns.
PR-AUC drops 2x for accounts unseen in training (0.160 seen vs 0.075 unseen).
This is the fundamental limitation of the tabular approach.

### Risk tier coverage (precision-anchored, test set)

Risk tiers are calibrated so each tier guarantees a minimum precision:
- **Critical** (>=15% precision): very high confidence -- immediate investigation
- **High** (>=5%): priority review queue
- **Medium-High** (>=2%): standard investigation queue
- **Medium** (>=0.5%, ~6x baseline): secondary screening
- **Low** (>=0.2%, ~2x baseline): automated flag
- **Very Low** (>=0.1%): watchlist only

See Section 12 for exact alert volumes, fraud counts, and the cumulative recall curve.

### Next step: GNN comparison

See Section 14 for the full motivation. The IBM benchmark reports a **95% improvement**
in F1 score for GNN over MLP on HI-Medium, driven primarily by multi-hop fraud signal
propagation to unseen accounts.

**Reference:** Altman et al. (2023). *Realistic Synthetic Financial Transactions for Anti-Money
Laundering Models.* NeurIPS 2023. arXiv:2306.16424.